# Entrenar la wake word "oye BB-8"

Entrena un modelo de [openWakeWord](https://github.com/dscripka/openWakeWord) para que el
BB-8 despierte con **"oye BB-8"** en lugar de "hey Jarvis". Usa voces sintéticas en español
(Piper), ruido y eco de cuartos reales, y ~2000 h de audio genérico como negativos.

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).** Sin GPU también corre, más lento.
2. Corre las celdas en orden. Cada paso se puede repetir: retoma donde se quedó.
3. Al final se descarga `oye_bb8.onnx`. Cópialo a la Pi en
   `partes/4-agente-movimiento/voz/modelos/oye_bb8.onnx` y reinicia `bb8-voz`.

Necesita unos 30 GB de disco (la máquina de Colab trae más). Guía completa:
[wiki/Wake-word-oye-BB-8.md](https://github.com/erickdsama/bb8/blob/main/wiki/Wake-word-oye-BB-8.md).

In [ ]:
# 1. Código y dependencias
RAMA = "main"
!git clone -q --depth 1 -b {RAMA} https://github.com/erickdsama/bb8 /content/bb8
%cd /content/bb8/partes/4-agente-movimiento/wakeword
!pip install -q -r requirements-entrenar.txt
!pip install -q --no-deps "openwakeword @ git+https://github.com/dscripka/openWakeWord@368c03716d1e92591906a84949bc477f3a834455"

In [ ]:
# 1b. (Opcional) Piper en GPU: genera las muestras varias veces más rápido.
# Si después falla la generación, reinicia el entorno y sáltate esta celda.
!pip uninstall -q -y onnxruntime && pip install -q onnxruntime-gpu

## 2. Tus grabaciones (opcional, pero ayuda mucho)

Graba entre 20 y 50 veces "oye BB-8" con tu voz, en el micrófono que usará el robot si
puedes (en la Pi: `arecord -r 16000 -c 1 -f S16_LE -d 2 oye_01.wav`), a distintas
distancias y tonos. Súbelas en un `.zip` con la celda de abajo. Si no tienes, sáltala.

In [ ]:
from google.colab import files
import zipfile, pathlib
subido = files.upload()
for nombre in subido:
    zipfile.ZipFile(nombre).extractall("/content/mis_grabaciones")
print(len(list(pathlib.Path("/content/mis_grabaciones").rglob("*.wav"))), "grabaciones")

In [ ]:
# 3. Datos: eco, ruido de fondo y ~17 GB de features negativos (tarda; se puede repetir)
TRABAJO = "/content/ww"
!python entrenar.py --trabajo {TRABAJO} datos

In [ ]:
# 4. Muestras: "oye BB-8" y frases parecidas con todas las voces de Piper en español
import pathlib
extra = "--mis-grabaciones /content/mis_grabaciones" if pathlib.Path("/content/mis_grabaciones").exists() else ""
!python entrenar.py --trabajo {TRABAJO} muestras {extra}

In [ ]:
# 5. Aumento de datos: les pone ruido y eco y saca las features de openWakeWord
!python entrenar.py --trabajo {TRABAJO} aumentar

In [ ]:
# 6. Entrenamiento (50 000 pasos) y exportación a oye_bb8.onnx
!python entrenar.py --trabajo {TRABAJO} modelo

In [ ]:
# 7. Prueba rápida. En positive_test y tus grabaciones, entre más detectadas mejor;
# en negative_test (frases parecidas), entre menos mejor.
!python probar.py {TRABAJO}/oye_bb8.onnx {TRABAJO}/salida/oye_bb8/positive_test | tail -3
!python probar.py {TRABAJO}/oye_bb8.onnx {TRABAJO}/salida/oye_bb8/negative_test | tail -3
if pathlib.Path("/content/mis_grabaciones").exists():
    !python probar.py {TRABAJO}/oye_bb8.onnx /content/mis_grabaciones | tail -3

In [ ]:
# 8. Descargar el modelo
from google.colab import files
files.download(f"{TRABAJO}/oye_bb8.onnx")